# ML3. Часть 4. Cross-validation / Кросс-валидация

В этом ноутбуке реализовано четвертое задание из README:

1. `K-Fold`, где `k` - входной параметр, функция возвращает список `train` и `test` индексов.
2. `Grouped K-Fold`, где `k` и `group_field` - входные параметры, функция возвращает список `train` и `test` индексов.
3. `Stratified K-fold`, где `k` и `stratify_field` - входные параметры, функция возвращает список `train` и `test` индексов.
4. `Time series split`, где `k` и `date_field` - входные параметры, функция возвращает список `train` и `test` индексов.

Все функции возвращают список пар `(train_indices, test_indices)`. Индексы являются **позиционными индексами**, то есть их нужно использовать через `.iloc`, например `df.iloc[train_indices]` и `df.iloc[test_indices]`. Такой подход похож на интерфейс кросс-валидации в `sklearn`.

## 1. Импорт библиотек и загрузка данных

Для реализации достаточно библиотек `pandas` и `numpy`, которые указаны в требованиях проекта. Методы ниже реализованы вручную, без использования готовых классов `KFold`, `GroupKFold`, `StratifiedKFold` и `TimeSeriesSplit` из `sklearn`, потому что по заданию нужно реализовать собственные методы.

In [36]:
import numpy as np
import pandas as pd

RANDOM_STATE = 21
DF_PATH = r".\data\train_df_processed_0.75.csv"

df = pd.read_csv(DF_PATH)
df["created"] = pd.to_datetime(df["created"])

print(f"Размер датасета: {df.shape}")
display(df.head())
display(df.dtypes)

Размер датасета: (36284, 24)


,bathrooms,bedrooms,price,interest,created,Elevator,HardwoodFloors,CatsAllowed,DogsAllowed,Doorman,...,Pre-War,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,NewConstruction,Terrace
0,1.0,0,2725,0,2016-05-26 15:01:42,1,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,1.0,3,5200,0,2016-06-16 05:42:00,1,1,0,0,0,...,0,0,0,1,0,0,0,0,0,0
2,1.0,2,4204,1,2016-04-20 02:00:53,1,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
3,1.0,1,3500,0,2016-04-22 05:35:23,1,0,0,0,1,...,0,0,1,0,1,0,0,1,0,0
4,1.0,1,2195,0,2016-05-20 01:22:06,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0


bathrooms                   float64
bedrooms                      int64
price                         int64
interest                      int64
created              datetime64[us]
Elevator                      int64
HardwoodFloors                int64
CatsAllowed                   int64
DogsAllowed                   int64
Doorman                       int64
Dishwasher                    int64
NoFee                         int64
LaundryinBuilding             int64
FitnessCenter                 int64
Pre-War                       int64
LaundryinUnit                 int64
RoofDeck                      int64
OutdoorSpace                  int64
DiningRoom                    int64
HighSpeedInternet             int64
Balcony                       int64
SwimmingPool                  int64
NewConstruction               int64
Terrace                       int64
dtype: object

## 2. Общие вспомогательные функции

Перед реализацией схем кросс-валидации добавим несколько проверок:

* `k` должен быть целым числом больше 1;
* датасет должен быть непустым;
* если функция использует поле `group_field`, `stratify_field` или `date_field`, такое поле должно существовать в DataFrame.

Также добавим функцию для краткого описания результата: она покажет, сколько фолдов получилось и какого размера `train` и `test` части.

In [37]:
def validate_dataframe(data: pd.DataFrame) -> None:
    """Проверяет, что на вход передан непустой pandas DataFrame."""
    if not isinstance(data, pd.DataFrame):
        raise TypeError("data должен быть pandas DataFrame")
    if data.empty:
        raise ValueError("data не должен быть пустым")


def validate_k(k: int, n_samples: int) -> None:
    """Проверяет корректность количества фолдов."""
    if not isinstance(k, int):
        raise TypeError("k должен быть целым числом")
    if k < 2:
        raise ValueError("k должен быть не меньше 2")
    if k > n_samples:
        raise ValueError("k не может быть больше количества объектов")


def validate_column(data: pd.DataFrame, column: str) -> None:
    """Проверяет, что колонка есть в DataFrame."""
    if column not in data.columns:
        raise KeyError(f"В DataFrame нет колонки {column!r}")


def describe_splits(splits, data: pd.DataFrame, title: str) -> None:
    """Печатает краткое описание списка train/test индексов."""
    print(title)
    print(f"Количество фолдов: {len(splits)}")
    for fold_number, (train_idx, test_idx) in enumerate(splits, start=1):
        intersection = np.intersect1d(train_idx, test_idx)
        print(
            f"Fold {fold_number}: "
            f"train={len(train_idx)}, test={len(test_idx)}, "
            f"intersection={len(intersection)}"
        )

## 3. K-Fold

`K-Fold` делит датасет на `k` примерно равных частей - фолдов. На каждой итерации один фолд используется как `test`, а остальные фолды объединяются в `train`.

Например, если `k = 5`, то будет 5 итераций:

* на первой итерации первый фолд - `test`, остальные - `train`;
* на второй итерации второй фолд - `test`, остальные - `train`;
* и так далее.

В этой реализации есть параметры `shuffle` и `random_state`. Если `shuffle=True`, объекты перед разделением перемешиваются детерминированно через `random_state`. Это полезно, если исходный порядок строк может быть неслучайным.

In [38]:
def make_k_fold_splits(
    data: pd.DataFrame,
    k: int,
    shuffle: bool = True,
    random_state: int = RANDOM_STATE,
) -> list[tuple[np.ndarray, np.ndarray]]:
    """Возвращает список пар позиционных индексов для K-Fold."""
    validate_dataframe(data)
    validate_k(k, len(data))

    indices = np.arange(len(data))

    if shuffle:
        rng = np.random.default_rng(random_state)
        indices = rng.permutation(indices)

    fold_indices = np.array_split(indices, k)
    splits = []

    for fold_number in range(k):
        test_indices = fold_indices[fold_number]
        train_indices = np.concatenate(
            [fold for i, fold in enumerate(fold_indices) if i != fold_number]
        )
        splits.append((train_indices, test_indices))

    return splits

In [39]:
k_fold_splits = make_k_fold_splits(df, k=5)
describe_splits(k_fold_splits, df, "K-Fold")

train_idx, test_idx = k_fold_splits[0]
display(df.iloc[train_idx].head())
display(df.iloc[test_idx].head())

K-Fold
Количество фолдов: 5
Fold 1: train=29027, test=7257, intersection=0
Fold 2: train=29027, test=7257, intersection=0
Fold 3: train=29027, test=7257, intersection=0
Fold 4: train=29027, test=7257, intersection=0
Fold 5: train=29028, test=7256, intersection=0


,bathrooms,bedrooms,price,interest,created,Elevator,HardwoodFloors,CatsAllowed,DogsAllowed,Doorman,...,Pre-War,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,NewConstruction,Terrace
14466,1.0,2,2660,0,2016-06-23 02:26:03,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
31828,2.0,3,3734,0,2016-05-14 03:07:51,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8565,1.0,1,2590,1,2016-05-14 04:08:08,0,1,0,0,0,...,0,0,0,1,0,0,0,0,0,0
11540,1.0,2,3595,2,2016-06-13 02:54:34,1,1,1,1,0,...,0,1,0,0,0,0,1,0,0,0
1204,1.0,1,2295,1,2016-05-06 04:51:40,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


,bathrooms,bedrooms,price,interest,created,Elevator,HardwoodFloors,CatsAllowed,DogsAllowed,Doorman,...,Pre-War,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,NewConstruction,Terrace
23895,2.0,4,5400,1,2016-06-21 04:26:15,1,1,1,1,1,...,0,0,1,0,1,0,0,0,0,0
19580,1.0,1,4000,0,2016-06-14 12:38:24,1,1,0,0,1,...,0,1,0,0,1,0,0,0,0,0
7124,2.0,3,4100,1,2016-06-03 05:41:15,1,1,0,0,0,...,0,1,0,0,0,0,0,0,0,0
19128,1.0,1,2885,1,2016-04-30 05:57:40,1,1,0,0,1,...,0,0,0,0,0,0,0,0,0,0
441,1.0,1,2485,0,2016-05-05 05:20:46,0,1,0,0,0,...,0,0,0,1,0,0,0,0,0,0


## 4. Grouped K-Fold

`Grouped K-Fold` нужен, когда строки датасета объединены в группы и нельзя допускать, чтобы одна и та же группа одновременно попала в `train` и `test`.

Примеры групп:

* один клиент с несколькими заявками;
* один пользователь с несколькими действиями;
* один объект недвижимости с несколькими объявлениями;
* один самолет, станок или другой физический объект с несколькими наблюдениями.

Главное правило: если группа попала в `test` на конкретном фолде, все строки этой группы должны быть только в `test`, а не частично в `train`.

В текущем предобработанном датасете нет явного `user_id` или `building_id`, поэтому для демонстрации используем `bedrooms` как технический пример `group_field`. В реальном проекте лучше использовать настоящую группу, например идентификатор клиента, здания или пользователя, если он есть в данных.

In [40]:
def make_group_k_fold_splits(
    data: pd.DataFrame,
    k: int,
    group_field: str,
) -> list[tuple[np.ndarray, np.ndarray]]:
    """Возвращает список пар позиционных индексов для Grouped K-Fold."""
    validate_dataframe(data)
    validate_k(k, len(data))
    validate_column(data, group_field)

    groups = data[group_field].to_numpy()
    unique_groups, group_counts = np.unique(groups, return_counts=True)

    if k > len(unique_groups):
        raise ValueError("k не может быть больше количества уникальных групп")

    # Сначала размещаем самые большие группы: так фолды получаются более сбалансированными по числу строк.
    sorted_group_order = np.argsort(group_counts)[::-1]
    sorted_groups = unique_groups[sorted_group_order]

    fold_groups = [[] for _ in range(k)]
    fold_sizes = np.zeros(k, dtype=int)

    for group in sorted_groups:
        group_size = np.sum(groups == group)
        target_fold = int(np.argmin(fold_sizes))
        fold_groups[target_fold].append(group)
        fold_sizes[target_fold] += group_size

    all_indices = np.arange(len(data))
    splits = []

    for groups_in_fold in fold_groups:
        test_mask = np.isin(groups, groups_in_fold)
        test_indices = all_indices[test_mask]
        train_indices = all_indices[~test_mask]
        splits.append((train_indices, test_indices))

    return splits

In [41]:
group_k_fold_splits = make_group_k_fold_splits(df, k=3, group_field="bedrooms")
describe_splits(group_k_fold_splits, df, "Grouped K-Fold по bedrooms")

for fold_number, (train_idx, test_idx) in enumerate(group_k_fold_splits, start=1):
    train_groups = set(df.iloc[train_idx]["bedrooms"])
    test_groups = set(df.iloc[test_idx]["bedrooms"])
    print(
        f"Fold {fold_number}: "
        f"train_groups={sorted(train_groups)}, "
        f"test_groups={sorted(test_groups)}, "
        f"group_intersection={sorted(train_groups & test_groups)}"
    )

Grouped K-Fold по bedrooms
Количество фолдов: 3
Fold 1: train=24447, test=11837, intersection=0
Fold 2: train=23958, test=12326, intersection=0
Fold 3: train=24163, test=12121, intersection=0
Fold 1: train_groups=[0, 2, 3, 4], test_groups=[1, 5, 6, 7, 8], group_intersection=[]
Fold 2: train_groups=[0, 1, 3, 5, 6, 7, 8], test_groups=[2, 4], group_intersection=[]
Fold 3: train_groups=[1, 2, 4, 5, 6, 7, 8], test_groups=[0, 3], group_intersection=[]


## 5. Stratified K-fold

`Stratified K-fold` нужен, когда важно сохранить распределение целевой переменной в каждом фолде. Это особенно полезно в задачах классификации, если классы несбалансированы.

В нашем датасете есть колонка `interest`, которую можно использовать как `stratify_field`. Идея такая:

1. отдельно берем индексы каждого класса;
2. перемешиваем индексы внутри класса детерминированно через `random_state`;
3. делим индексы каждого класса на `k` частей;
4. для каждого фолда объединяем соответствующие части всех классов.

Так в каждом `test` фолде сохраняются примерно те же доли классов, что и в исходном датасете.

In [42]:
def make_stratified_k_fold_splits(
    data: pd.DataFrame,
    k: int,
    stratify_field: str,
    shuffle: bool = True,
    random_state: int = RANDOM_STATE,
) -> list[tuple[np.ndarray, np.ndarray]]:
    """Возвращает список пар позиционных индексов для Stratified K-Fold."""
    validate_dataframe(data)
    validate_k(k, len(data))
    validate_column(data, stratify_field)

    values = data[stratify_field].to_numpy()
    all_indices = np.arange(len(data))
    rng = np.random.default_rng(random_state)

    fold_parts = [[] for _ in range(k)]

    for class_value in np.unique(values):
        class_indices = all_indices[values == class_value]

        if len(class_indices) < k:
            raise ValueError(
                f"Класс {class_value!r} содержит {len(class_indices)} объектов, "
                f"что меньше k={k}"
            )

        if shuffle:
            class_indices = rng.permutation(class_indices)

        class_folds = np.array_split(class_indices, k)

        for fold_number, class_fold in enumerate(class_folds):
            fold_parts[fold_number].append(class_fold)

    splits = []

    for fold_number in range(k):
        test_indices = np.concatenate(fold_parts[fold_number])
        train_indices = np.setdiff1d(all_indices, test_indices, assume_unique=False)
        splits.append((train_indices, test_indices))

    return splits

In [43]:
stratified_k_fold_splits = make_stratified_k_fold_splits(df, k=5, stratify_field="interest")
describe_splits(stratified_k_fold_splits, df, "Stratified K-Fold по interest")

print("Распределение классов в полном датасете:")
display(df["interest"].value_counts(normalize=True).sort_index().rename("full_share"))

for fold_number, (_, test_idx) in enumerate(stratified_k_fold_splits, start=1):
    print(f"Fold {fold_number}, распределение классов в test:")
    display(df.iloc[test_idx]["interest"].value_counts(normalize=True).sort_index().rename("test_share"))

Stratified K-Fold по interest
Количество фолдов: 5
Fold 1: train=29026, test=7258, intersection=0
Fold 2: train=29026, test=7258, intersection=0
Fold 3: train=29028, test=7256, intersection=0
Fold 4: train=29028, test=7256, intersection=0
Fold 5: train=29028, test=7256, intersection=0
Распределение классов в полном датасете:


interest
0    0.696643
1    0.228944
2    0.074413
Name: full_share, dtype: float64

Fold 1, распределение классов в test:


interest
0    0.696611
1    0.228989
2    0.074401
Name: test_share, dtype: float64

Fold 2, распределение классов в test:


...

interest
0    0.696665
1    0.228914
2    0.074421
Name: test_share, dtype: float64

Fold 4, распределение классов в test:


interest
0    0.696665
1    0.228914
2    0.074421
Name: test_share, dtype: float64

Fold 5, распределение классов в test:


interest
0    0.696665
1    0.228914
2    0.074421
Name: test_share, dtype: float64

## 6. Time series split

`Time series split` используется, когда данные имеют временную зависимость. В таком случае нельзя случайно перемешивать строки: это может привести к утечке будущей информации в обучение.

По описанию из README:

* сначала сортируем данные по дате;
* задаем `k` как количество временных частей;
* для первой модели берем `1/k` данных в `train` и следующие `1/k` в `test`;
* для второй модели расширяем `train` до `2/k`, а `test` сдвигаем на следующий блок;
* продолжаем так до конца.

Поэтому при `k` временных частях получается `k - 1` train/test разбиений.

In [44]:
def make_time_series_splits(
    data: pd.DataFrame,
    k: int,
    date_field: str,
) -> list[tuple[np.ndarray, np.ndarray]]:
    """Возвращает список пар позиционных индексов для Time series split."""
    validate_dataframe(data)
    validate_k(k, len(data))
    validate_column(data, date_field)

    dates = pd.to_datetime(data[date_field])
    sorted_positions = np.argsort(dates.to_numpy())
    time_blocks = np.array_split(sorted_positions, k)

    splits = []

    for test_block_number in range(1, k):
        train_indices = np.concatenate(time_blocks[:test_block_number])
        test_indices = time_blocks[test_block_number]
        splits.append((train_indices, test_indices))

    return splits

In [45]:
time_series_splits = make_time_series_splits(df, k=5, date_field="created")
describe_splits(time_series_splits, df, "Time series split по created")

for fold_number, (train_idx, test_idx) in enumerate(time_series_splits, start=1):
    train_dates = df.iloc[train_idx]["created"]
    test_dates = df.iloc[test_idx]["created"]
    print(
        f"Fold {fold_number}: "
        f"train {train_dates.min()} - {train_dates.max()}, "
        f"test {test_dates.min()} - {test_dates.max()}"
    )

Time series split по created
Количество фолдов: 4
Fold 1: train=7257, test=7257, intersection=0
Fold 2: train=14514, test=7257, intersection=0
Fold 3: train=21771, test=7257, intersection=0
Fold 4: train=29028, test=7256, intersection=0
Fold 1: train 2016-04-01 22:12:41 - 2016-04-19 03:40:22, test 2016-04-19 03:40:26 - 2016-05-07 02:52:37
Fold 2: train 2016-04-01 22:12:41 - 2016-05-07 02:52:37, test 2016-05-07 02:52:51 - 2016-05-24 14:55:17
Fold 3: train 2016-04-01 22:12:41 - 2016-05-24 14:55:17, test 2016-05-24 14:57:06 - 2016-06-12 08:29:35
Fold 4: train 2016-04-01 22:12:41 - 2016-06-12 08:29:35, test 2016-06-12 08:29:43 - 2016-06-29 21:41:47


## 7. Итоговая проверка формата результата

Все реализованные функции возвращают один и тот же тип результата:

`list[tuple[np.ndarray, np.ndarray]]`

То есть список фолдов, где каждый фолд - это пара:

* `train_indices` - позиционные индексы строк для обучения;
* `test_indices` - позиционные индексы строк для проверки.

Дальше эти индексы можно использовать так:

`df_train = df.iloc[train_indices]`

`df_test = df.iloc[test_indices]`

In [46]:
all_split_examples = {
    "K-Fold": k_fold_splits,
    "Grouped K-Fold": group_k_fold_splits,
    "Stratified K-Fold": stratified_k_fold_splits,
    "Time series split": time_series_splits,
}

for split_name, splits in all_split_examples.items():
    first_train_idx, first_test_idx = splits[0]
    print(split_name)
    print("Тип результата:", type(splits))
    print("Тип train индексов:", type(first_train_idx), "размер:", first_train_idx.shape)
    print("Тип test индексов:", type(first_test_idx), "размер:", first_test_idx.shape)
    print("Пример df_train shape:", df.iloc[first_train_idx].shape)
    print("Пример df_test shape:", df.iloc[first_test_idx].shape)
    print("-" * 60)

K-Fold
Тип результата: <class 'list'>
Тип train индексов: <class 'numpy.ndarray'> размер: (29027,)
Тип test индексов: <class 'numpy.ndarray'> размер: (7257,)
Пример df_train shape: (29027, 24)
Пример df_test shape: (7257, 24)
------------------------------------------------------------
Grouped K-Fold
Тип результата: <class 'list'>
Тип train индексов: <class 'numpy.ndarray'> размер: (24447,)
Тип test индексов: <class 'numpy.ndarray'> размер: (11837,)
Пример df_train shape: (24447, 24)
Пример df_test shape: (11837, 24)
------------------------------------------------------------
Stratified K-Fold
Тип результата: <class 'list'>
Тип train индексов: <class 'numpy.ndarray'> размер: (29026,)
Тип test индексов: <class 'numpy.ndarray'> размер: (7258,)
Пример df_train shape: (29026, 24)
Пример df_test shape: (7258, 24)
------------------------------------------------------------
Time series split
Тип результата: <class 'list'>
Тип train индексов: <class 'numpy.ndarray'> размер: (7257,)
Тип test 

## 8. Краткие выводы

* `K-Fold` подходит для обычной кросс-валидации, когда нет групп и временной зависимости.
* `Grouped K-Fold` нужен, когда нельзя разделять объекты одной группы между `train` и `test`.
* `Stratified K-fold` нужен для сохранения распределения целевой переменной в каждом фолде.
* `Time series split` нужен для данных с временной структурой: модель обучается на прошлом и проверяется на будущем.

Выбор схемы кросс-валидации зависит не от удобства кода, а от природы данных и от того, как модель будет использоваться в production.